# Data Collection for Multimodal Phishing Detection

This notebook collects the dataset for training the multimodal phishing detection model.

## Dataset Configuration
- **Total Samples**: 1,697 (matching research paper)
- **Legitimate**: 1,147 URLs (68%)
- **Phishing**: 550 URLs (32%)
- **Sources**: PhishTank, URLhaus, Tranco

In [ ]:
# Install required packages
!pip install requests pandas tqdm beautifulsoup4

In [ ]:
# Mount Google Drive (for saving data)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import libraries
import os
import pandas as pd
import requests
import numpy as np
from tqdm import tqdm
import time
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configuration
PHISHTANK_API_URL = "https://data.phishtank.com/data/online-valid.json"
URLHAUS_API_URL = "https://urlhaus.abuse.ch/downloads/csv_recent/"
TRANCO_URL = "https://tranco-list.eu/1000000.csv"

DATASET_SIZE = 1697
LEGITIMATE_SAMPLES = 1147
PHISHING_SAMPLES = 550

# Google Drive path
DRIVE_PATH = "/content/drive/MyDrive/Multimodal_Phishing_Detection"
DATA_PATH = os.path.join(DRIVE_PATH, "data")
os.makedirs(DATA_PATH, exist_ok=True)

print(f"Data will be saved to: {DATA_PATH}")

## 1. Collect Phishing URLs

In [ ]:
def collect_phishing_urls():
    """Collect phishing URLs from PhishTank and URLhaus."""
    phishing_urls = []
    
    # Collect from PhishTank
    try:
        logger.info("Fetching from PhishTank...")
        response = requests.get(PHISHTANK_API_URL, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        
        if isinstance(data, list):
            for item in data[:PHISHING_SAMPLES]:
                if 'url' in item:
                    phishing_urls.append({
                        'url': item['url'],
                        'source': 'phishtank',
                        'verified': item.get('verified', 'yes')
                    })
        
        logger.info(f"Collected {len(phishing_urls)} URLs from PhishTank")
        
    except Exception as e:
        logger.error(f"Error fetching from PhishTank: {e}")
    
    # If we need more URLs, try URLhaus
    if len(phishing_urls) < PHISHING_SAMPLES:
        try:
            logger.info("Fetching from URLhaus...")
            response = requests.get(URLHAUS_API_URL, timeout=30)
            response.raise_for_status()
            
            lines = response.text.split('\n')
            for line in lines[1:]:  # Skip header
                if line.strip() and len(phishing_urls) < PHISHING_SAMPLES:
                    parts = line.split(',')
                    if len(parts) > 2:
                        phishing_urls.append({
                            'url': parts[2],
                            'source': 'urlhaus',
                            'verified': 'yes'
                        })
            
            logger.info(f"Total phishing URLs collected: {len(phishing_urls)}")
            
        except Exception as e:
            logger.error(f"Error fetching from URLhaus: {e}")
    
    return phishing_urls[:PHISHING_SAMPLES]

# Collect phishing URLs
phishing_urls = collect_phishing_urls()
print(f"Collected {len(phishing_urls)} phishing URLs")
print(phishing_urls[:3])

## 2. Collect Legitimate URLs

In [ ]:
def collect_legitimate_urls():
    """Collect legitimate URLs from Tranco top list."""
    legitimate_urls = []
    
    try:
        logger.info("Fetching from Tranco...")
        response = requests.get(TRANCO_URL, timeout=30)
        response.raise_for_status()
        
        lines = response.text.split('\n')
        for line in lines[:LEGITIMATE_SAMPLES + 1]:  # +1 for header
            if line.strip() and ',' in line:
                parts = line.split(',')
                if len(parts) >= 2 and parts[1].replace('.', '').isalnum():
                    legitimate_urls.append({
                        'url': parts[1],
                        'source': 'tranco',
                        'rank': int(parts[0])
                    })
        
        logger.info(f"Collected {len(legitimate_urls)} legitimate URLs")
        
    except Exception as e:
        logger.error(f"Error fetching from Tranco: {e}")
        
        # Fallback to common legitimate domains
        fallback_domains = [
            'google.com', 'facebook.com', 'youtube.com', 'amazon.com',
            'microsoft.com', 'apple.com', 'netflix.com', 'instagram.com',
            'twitter.com', 'linkedin.com', 'github.com', 'stackoverflow.com',
            'wikipedia.org', 'reddit.com', 'medium.com', 'quora.com'
        ]
        
        legitimate_urls = [
            {'url': domain, 'source': 'fallback', 'rank': i+1}
            for i, domain in enumerate(fallback_domains[:LEGITIMATE_SAMPLES])
        ]
        
        logger.info(f"Used fallback domains: {len(legitimate_urls)}")
    
    return legitimate_urls[:LEGITIMATE_SAMPLES]

# Collect legitimate URLs
legitimate_urls = collect_legitimate_urls()
print(f"Collected {len(legitimate_urls)} legitimate URLs")
print(legitimate_urls[:3])

## 3. Save Dataset

In [ ]:
# Create DataFrames
phishing_df = pd.DataFrame(phishing_urls)
phishing_df['label'] = 1  # Phishing = 1

legitimate_df = pd.DataFrame(legitimate_urls)
legitimate_df['label'] = 0  # Legitimate = 0

# Combine datasets
combined_df = pd.concat([phishing_df, legitimate_df], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save datasets
phishing_df.to_csv(os.path.join(DATA_PATH, 'phishing_urls.csv'), index=False)
legitimate_df.to_csv(os.path.join(DATA_PATH, 'legitimate_urls.csv'), index=False)
combined_df.to_csv(os.path.join(DATA_PATH, 'combined_urls.csv'), index=False)

print(f"Dataset saved to Google Drive: {DATA_PATH}")
print(f"\nDataset Summary:")
print(f"  - Total URLs: {len(combined_df)}")
print(f"  - Phishing: {len(phishing_df)}")
print(f"  - Legitimate: {len(legitimate_df)}")
print(f"\nSample data:")
print(combined_df.head())

## 4. Verify Dataset

Let's verify the dataset matches the research paper specifications.

In [ ]:
# Dataset verification
print("Dataset Verification:")
print(f"  - Expected total: {DATASET_SIZE}")
print(f"  - Actual total: {len(combined_df)}")
print(f"  - Expected phishing: {PHISHING_SAMPLES}")
print(f"  - Actual phishing: {len(phishing_df)}")
print(f"  - Expected legitimate: {LEGITIMATE_SAMPLES}")
print(f"  - Actual legitimate: {len(legitimate_df)}")

# Check class distribution
class_dist = combined_df['label'].value_counts()
print(f"\nClass Distribution:")
print(f"  - Legitimate (0): {class_dist[0]} ({class_dist[0]/len(combined_df)*100:.1f}%)")
print(f"  - Phishing (1): {class_dist[1]} ({class_dist[1]/len(combined_df)*100:.1f}%)")

# Check sources
print(f"\nPhishing Sources:")
print(phishing_df['source'].value_counts())
print(f"\nLegitimate Sources:")
print(legitimate_df['source'].value_counts())

## 5. Next Steps

The dataset is now ready for preprocessing. The next steps are:

1. **Text Preprocessing**: Run the text preprocessing notebook
2. **Visual Preprocessing**: Run the visual preprocessing notebook
3. **Model Training**: Run the training notebook

The dataset is saved in Google Drive and can be accessed from the next notebooks.